In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint


from src.preprocessing import final_pipeline

In [2]:
def load_data():
    path = Path("dataset/used_cars.csv")

    return pd.read_csv(path)

In [3]:
used_cars_data = load_data()

Splitting into train and test sets with stratification by price

In [4]:
used_cars_data["price"] = (used_cars_data["price"].str.replace(r"[$,]", "", regex=True).astype(int))  # removing non-numeric characters and converting to int

used_cars_data["price_cat"] = pd.qcut(used_cars_data["price"], q=5, labels=[1, 2, 3, 4, 5])

strat_train_set, strat_test_set = train_test_split(used_cars_data, test_size=0.2, stratify=used_cars_data["price_cat"], random_state=35)

Removing the created category column from the train and test sets

In [5]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("price_cat", axis=1, inplace=True)

Splitting a dataset by features and label

In [6]:
train = strat_train_set.drop("price", axis=1)  # drop column with target value and create copy of data

train_labels = strat_train_set["price"].copy()  # create a copy of only the target value's data

# Baseline

naive RMSE

In [7]:
print("naive RMSE:", train_labels.std())

naive RMSE: 59801.351583500764


naive MAE

In [29]:
naive_mae = (train_labels - train_labels.median()).abs().mean()

print("naive MAE:", round(naive_mae, 4), "\nmedian price:", train_labels.median())

naive MAE: 25906.6296 
median price: 31000.0


# Linear Regression model

In [9]:
lin_reg = TransformedTargetRegressor(
    regressor=make_pipeline(final_pipeline, LinearRegression()),
    func=np.log1p,
    inverse_func=np.expm1
)

lin_reg.fit(train, train_labels)

lin_price_preds = lin_reg.predict(train)  # model's predictions

lin_price_preds[:5].round(-2)  # first 5 predictions rounded to the nearest hundred

array([38300., 14300.,  8500., 46000., 53500.])

In [10]:
train_labels.iloc[:5].values  # first 5 real values

array([21000,  9999,  6250, 42000, 57352])

RMSE as performance measure for a Linear Regression model

In [11]:
lin_rmse = root_mean_squared_error(train_labels, lin_price_preds)

print("train RMSE: ", lin_rmse)

train RMSE:  44533.000101740254


In [12]:
lin_cv_rmses = -cross_val_score(lin_reg, train, train_labels, scoring="neg_root_mean_squared_error", cv=10)

pd.Series(lin_cv_rmses).describe()

count        10.000000
mean      40140.344879
std       26560.676252
min       17056.183697
25%       27430.607304
50%       29058.659370
75%       44676.286543
max      109934.652900
dtype: float64

# Regression Decision Tree model

In [13]:
tree_reg = TransformedTargetRegressor(
    regressor=make_pipeline(final_pipeline, DecisionTreeRegressor(random_state=42)),
    func=np.log1p,
    inverse_func=np.expm1
)

tree_reg.fit(train, train_labels)

tree_price_preds = tree_reg.predict(train)  # model's predictions

RMSE

In [14]:
tree_rmse = root_mean_squared_error(train_labels, tree_price_preds)

print("train RMSE: ", tree_rmse)  # expected a value close to 0

train RMSE:  3.8411670095210344e-11


In [15]:
tree_cv_rmses = -cross_val_score(tree_reg, train, train_labels, scoring="neg_root_mean_squared_error", cv=10)

pd.Series(tree_cv_rmses).describe()

count       10.000000
mean     49737.571695
std      27689.886558
min      21236.349623
25%      28896.534116
50%      38276.854943
75%      77486.557889
max      90622.316476
dtype: float64

# Random Forest model

In [16]:
forest_reg = TransformedTargetRegressor(
    regressor=make_pipeline(final_pipeline, RandomForestRegressor(random_state=42)),
    func=np.log1p,
    inverse_func=np.expm1
)

RMSE

In [17]:
forest_cv_rmses = -cross_val_score(forest_reg, train, train_labels, scoring="neg_root_mean_squared_error", cv=10)

pd.Series(forest_cv_rmses).describe()

count        10.000000
mean      28622.313965
std       25835.657226
min       11275.290443
25%       16698.517830
50%       21674.752550
75%       24964.096891
max      100378.795742
dtype: float64

MAE

In [70]:
forest_cv_mae = -cross_val_score(forest_reg, train, train_labels, scoring="neg_mean_absolute_error", cv=10)

pd.Series(forest_cv_mae).describe()

count       10.000000
mean      9910.572138
std       2631.027860
min       7403.549146
25%       8497.566951
50%       9557.776508
75%       9772.419199
max      16866.513962
dtype: float64

In [ ]:
forest_cv_preds = cross_val_predict(forest_reg, train, train_labels, cv=10)

rel_err = (forest_cv_preds - train_labels).abs() / train_labels

print("Median Absolute Percentage Error:", round(rel_err.median(), 3))

In [ ]:
rel_err.groupby(pd.qcut(train_labels, 5)).median()

In [65]:
p = "regressor__randomforestregressor__"
param_distributions = {
    p + "n_estimators": randint(200, 400),
    p + "max_features": [0.3, 0.5, 0.7, 1.0],
    p + "min_samples_leaf": randint(5, 16),
    p + "max_depth": [10, 15, 20, 25],
}

rnd_search = RandomizedSearchCV(
    forest_reg, param_distributions, n_iter=20, cv=10,
    scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1
)

rnd_search.fit(train, train_labels)

,estimator,TransformedTa..._state=42))]))
,param_distributions,"{'regressor__randomforestregressor__max_depth': [10, 15, ...], 'regressor__randomforestregressor__max_features': [0.3, 0.5, ...], 'regressor__randomfores...essor__min_samples_leaf': <scipy.stats....002416BB18440>, 'regressor__randomforestregressor__n_estimators': <scipy.stats....002416BB190D0>}"
,n_iter,20
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [66]:
print(rnd_search.best_params_)

{'regressor__randomforestregressor__max_depth': 10, 'regressor__randomforestregressor__max_features': 1.0, 'regressor__randomforestregressor__min_samples_leaf': 6, 'regressor__randomforestregressor__n_estimators': 208}


RMSE

In [67]:
print(-rnd_search.best_score_)

32453.430701819052


Importance of features

In [50]:
best_pipe = rnd_search.best_estimator_.regressor_

names = best_pipe[0].transform(train).columns

importances = pd.Series(best_pipe[-1].feature_importances_, index=names)

In [35]:
print(importances.sort_values(ascending=False).head(10))

NameError: name 'importances' is not defined

In [19]:
final_model = forest_reg.fit(train, train_labels)

X_test = strat_test_set.drop("price", axis=1)
y_test = strat_test_set["price"].copy()

final_predictions = final_model.predict(X_test)

In [20]:
print("test RMSE:", root_mean_squared_error(y_test, final_predictions))

test RMSE: 120106.62062829906


In [21]:
print("test MAE:", mean_absolute_error(y_test, final_predictions))

test MAE: 15002.797207294505


In [22]:
errors = (final_predictions - y_test).abs()

print(X_test.assign(price=y_test, pred=final_predictions, err=errors)
        .nlargest(10, "err")[["brand", "model", "model_year", "milage", "price", "pred", "err"]])

              brand               model  model_year      milage    price  \
693        Maserati   Quattroporte Base        2005  32,000 mi.  2954083   
3046        Porsche     Carrera GT Base        2005   4,400 mi.  1599000   
1615    Rolls-Royce             Phantom        2023   1,560 mi.   599995   
203            Ford                  GT        2005   1,897 mi.   429998   
3611  Mercedes-Benz       AMG G 63 Base        2021   7,856 mi.   425000   
1148        Porsche               911 R        2016     719 mi.   489000   
1061          Dodge           Viper GTC        2017   1,389 mi.   489995   
1597        Ferrari  812 Superfast Base        2020   1,575 mi.   469900   
3697        Porsche          911 GT3 RS        2019  15,509 mi.   229900   
939     Rolls-Royce          Ghost Base        2016   8,755 mi.   166900   

               pred           err  
693    24336.893114  2.929746e+06  
3046  126673.690470  1.472326e+06  
1615   67622.670571  5.323723e+05  
203    86271.662370

In [32]:
print("RMSE percentage for the 10 largest values:", round((errors**2).nlargest(10).sum() / (errors**2).sum(), 4))

RMSE percentage for the 10 largest values: 0.9891
